# Brawl Stars Draft Agent — Training Workflow

This notebook trains a **joint policy+value network** that powers the draft recommendation agent. It covers every step from raw season data to a fully trained model.

---

### What gets built

| Step | Artifact | Purpose |
|------|----------|---------|
| 1 | `fm_model.pkl` | Factorization Machine — estimates win probability for a completed 6-pick draft. This is the ground truth signal that labels all training data. |
| 2 | `matchup_db.pkl` | Empirical counter/synergy lookup tables — used by MCTS to model opponent picks during self-play. |
| 3 | `self_play/` | Replay buffer — self-play games where two MCTS agents draft against each other. Each pick becomes a training record. |
| 4 | `joint_net.pkl` | Joint policy+value network — the final artifact. Its **policy head** guides MCTS search; its **value head** replaces random rollouts, making recommendations ~5× faster. |

---

### Two ways to use this notebook

**Starting fresh** — run all cells from top to bottom.

**Resuming** — run Section 0 (all three cells), check the status output, then jump directly to the section flagged with `▶ Next:`. Each section reloads its own artifacts from disk, so skipping completed sections is safe.

The self-play loop (Sections 3–4) is designed to be run multiple times. Each pass generates more games, trains a better network, and overwrites `joint_net.pkl` if the new version wins a head-to-head evaluation.

---
## Section 0 — Environment Setup

Run these three cells every time you open the notebook, regardless of where you plan to resume. They configure paths, import modules, and show you what's already done.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  0.1 — GLOBAL CONFIGURATION                                 ║
# ║  Edit the values below before running anything else.        ║
# ╚══════════════════════════════════════════════════════════════╝

from pathlib import Path

# Season key — must match an entry in src/data_prep.py::SEASON_CONFIGS.
# Available seasons: "s42", "s48", "s49"
SEASON = "s49"

# Path to the SQLite database for this season.
# This is where raw ranked match data lives.
REPO_ROOT = Path.cwd().parent  # assumes the notebook is run from notebooks/
DB_PATH = REPO_ROOT / "season49" / "v1_clean.db"

# Directory where all trained artifacts are saved and loaded from.
# Sub-directories (self_play/, policy/) are created automatically.
DATA_DIR = REPO_ROOT / "data" / SEASON

# PyTorch device. "cuda" if you have a GPU, otherwise "cpu".
# CPU is fine — FM training is the only step that meaningfully benefits from GPU.
DEVICE = "cpu"

# ── Derived paths (do not edit) ───────────────────────────────────────────────
SP_DIR = DATA_DIR / "self_play"   # replay buffer directory
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Season  : {SEASON}")
print(f"DB      : {DB_PATH}")
print(f"Data dir: {DATA_DIR}")
print(f"Device  : {DEVICE}")

In [ ]:
# 0.2 — Imports & Dependency Check
#
# Adds src/ to the Python path so all project modules are importable,
# then imports everything used across the notebook.

import sys

# Make src/ importable from within the notebooks/ directory.
SRC_DIR = str(REPO_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import numpy as np
import torch

from data_prep import SEASON_CONFIGS, build_game_dataset
from feature_engineering import (
    FeatureSchema,
    build_schema,
    build_feature_matrix,
    chronological_split,
)
from fm_model import FMInference, train_fm
from fm_evaluate import check_fm_calibration
from fm_integration import FMEvaluator
from matchup_db import MatchupDB
from self_play import ReplayBuffer, run_self_play_batch, load_map_mode_pairs
from joint_net import JointNetInference, train_joint
from recommend import _run_mcts
from workflow_utils import check_pipeline_status

# ── Checks ────────────────────────────────────────────────────────────────────

# Verify the season config exists
assert SEASON in SEASON_CONFIGS, (
    f"Unknown season '{SEASON}'. Add it to SEASON_CONFIGS in src/data_prep.py "
    f"or choose from: {list(SEASON_CONFIGS)}"
)

# Verify the database file is reachable
assert DB_PATH.exists(), (
    f"Database not found: {DB_PATH}\n"
    "Check DB_PATH in cell 0.1."
)

# Report PyTorch and CUDA status
cuda_available = torch.cuda.is_available()
mps_available  = getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()

print(f"PyTorch  : {torch.__version__}")
print(f"NumPy    : {np.__version__}")
print(f"CUDA     : {'available' if cuda_available else 'not available'}")
print(f"MPS      : {'available' if mps_available else 'not available'}")
print(f"Device   : {DEVICE}")

if DEVICE == "cuda" and not cuda_available:
    print("\n⚠  WARNING: DEVICE='cuda' but CUDA is not available. "
          "Change DEVICE to 'cpu' in cell 0.1.")
elif DEVICE == "cpu" and (cuda_available or mps_available):
    accel = "CUDA" if cuda_available else "MPS"
    print(f"\n  Tip: {accel} is available. Setting DEVICE='{accel.lower()}' "
          f"in cell 0.1 will speed up FM training.")

print("\nAll imports OK.")

In [ ]:
# 0.3 — Pipeline Status Check
#
# Scans DATA_DIR for existing artifacts and shows a checklist.
# The line marked ▶ tells you which section to jump to.

status = check_pipeline_status(DATA_DIR)

---
## Section 1 — Factorization Machine

**Skip this section** if `fm_model.pkl` already exists in `DATA_DIR` (shown as `[✓]` in the status check above).

---

The Factorization Machine (FM) is the foundation of the entire system. Given a completed 6-pick draft — three brawlers per team, a map, a mode, and a skill level — it outputs a single number: the estimated probability that your team wins.

Every piece of training data produced in self-play (Section 3) is labeled using FM predictions. If the FM is inaccurate, those labels are noisy, and the joint network trains on bad signal. **The FM must pass its quality check before you proceed.**

### How it works

The FM learns two things simultaneously:
- **Linear effects** — each individual brawler's strength on each map/mode
- **Pairwise interactions** — counter relationships and synergies between any two brawlers, encoded as a dot product of learned embedding vectors

This gives it the ability to capture "Brawler A counters Brawler B" without needing an explicit counter table. The embeddings are the same ones used later to initialize the joint network's input encoding.

### Chronological train/val split

Matches are split 80/20 **by time**, not randomly. The model is trained on earlier matches and evaluated on more recent ones. This mirrors real deployment: the agent will always face future data. Random splitting would inflate validation metrics by letting the model see "future" brawler interactions during training.

### What to expect

- **Val log-loss < 0.685** — the random-guess baseline is 0.693. Anything below 0.685 means the FM has learned meaningful signal. Our typical result is ~0.667.
- **Val AUC > 0.55** — random is 0.500. A modest AUC (~0.62) is expected and fine; Brawl Stars drafts have high inherent variance.
- **Val Brier < 0.245** — random baseline is 0.250. Measures calibration + accuracy together.

A well-trained FM also provides accurate win-probability *estimates*, not just rankings. This calibration matters because the joint network's value head is trained to reproduce FM outputs — if those outputs are systematically biased, the value head inherits the bias.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  1.1 — FM HYPERPARAMETERS                                   ║
# ║  Tune these before running cells 1.2–1.4.                   ║
# ╚══════════════════════════════════════════════════════════════╝

# --- FM Architecture ---
FM_K = 32
# Embedding dimension for pairwise interaction factors.
# Higher k → more expressive (captures finer brawler interactions) but slower to train.
# Recommended range: 16–64. Start with 32; try 16 if training is too slow,
# or 64 if the quality check in cell 1.5 is marginal.

# --- Training Schedule ---
FM_LR           = 1e-3
# Adam learning rate. The default works well; reduce to 3e-4 if loss is unstable.

FM_WEIGHT_DECAY = 1e-4
# L2 regularization applied uniformly to all parameters.
# Increase to 1e-3 if val loss is consistently worse than train loss (overfitting).

FM_BATCH_SIZE   = 4096
# Mini-batch size. Larger → faster epochs but slightly noisier gradients.
# Reduce to 2048 if you run out of memory on CPU.

FM_MAX_EPOCHS   = 50
# Hard training cap. Early stopping (FM_PATIENCE) will usually trigger well before this.

FM_PATIENCE     = 5
# Stop training if val loss does not improve for this many consecutive epochs.
# The FM typically converges in 10–20 epochs.

# --- Data Quality Filters ---
AVG_ELO_MIN = 10
# Minimum average ELO for a match to be included.
# Matches below this are low-quality ranked games unlikely to reflect drafting skill.

AVG_ELO_MAX = 23
# Maximum average ELO. The scale tops out around 25; values above 23 are sparse.
# Filtering the upper tail removes outlier match conditions.

print("FM hyperparameters set:")
print(f"  k={FM_K}  lr={FM_LR}  weight_decay={FM_WEIGHT_DECAY}  "
      f"batch_size={FM_BATCH_SIZE}  max_epochs={FM_MAX_EPOCHS}  patience={FM_PATIENCE}")
print(f"  ELO filter: [{AVG_ELO_MIN}, {AVG_ELO_MAX}]")

In [ ]:
# 1.2 — Load & Preprocess Data
#
# Loads the season's SQLite database, applies quality filters, and expands
# set-level records (one row per match) into game-level rows (one row per
# individual game within a match). This is the raw training data.

df_games, vocab = build_game_dataset(
    db_path=DB_PATH,
    elo_min=AVG_ELO_MIN,
    elo_max=AVG_ELO_MAX,
)

# ── Summary ───────────────────────────────────────────────────────────────────
n_games      = len(df_games)
n_maps       = df_games["map"].nunique()
n_modes      = df_games["mode"].nunique()
n_brawlers   = len(vocab)
win_rate     = df_games["team1_wins"].mean()
balance_ok   = abs(win_rate - 0.5) < 0.02

print(f"\nDataset loaded:")
print(f"  Games      : {n_games:>10,}")
print(f"  Brawlers   : {n_brawlers:>10,}  (vocabulary size)")
print(f"  Maps       : {n_maps:>10,}")
print(f"  Modes      : {n_modes:>10,}  {sorted(df_games['mode'].unique())}")
print(f"  Win rate   : {win_rate:>10.4f}  (team1; expected ~0.50)")
print(f"  Balance    : {'OK ✓' if balance_ok else 'WARNING ✗ — investigate before training'}")

if not balance_ok:
    print(f"\n  ⚠  Win rate {win_rate:.4f} deviates >2% from 0.50. This suggests a "
          "labelling bias in the data. Check the ELO filters or raw data.")

In [ ]:
# 1.3 — Build Feature Schema
#
# The feature schema defines the sparse encoding used by the FM (and later by
# the joint network). It maps brawler/map/mode names to column indices in the
# feature matrix. Building it here also saves it to disk so that subsequent
# train_fm() and joint network training calls can reuse it without rebuilding.
#
# Feature layout (one row per game, 9 non-zero entries):
#   [t1_brawler_0, t1_brawler_1, t1_brawler_2]  ← my team (3 one-hot)
#   [t2_brawler_0, t2_brawler_1, t2_brawler_2]  ← opponent team (3 one-hot)
#   [map_index]                                  ← map (1 one-hot)
#   [mode_index]                                 ← mode (1 one-hot)
#   [skill_ns]                                   ← continuous normalized skill score
#
# Team-swap augmentation doubles the row count: for every game, a second row is
# added with teams swapped and the label inverted (1 − win). This enforces the
# constraint that P(A beats B) + P(B beats A) = 1.

SCHEMA_PATH = DATA_DIR / "feature_schema.pkl"

if SCHEMA_PATH.exists():
    schema = FeatureSchema.load(SCHEMA_PATH)
    print(f"Loaded existing schema from {SCHEMA_PATH}")
else:
    schema = build_schema(df_games)
    schema.save(SCHEMA_PATH)
    print(f"Built and saved schema to {SCHEMA_PATH}")

# Estimate feature matrix dimensions without building it (expensive)
n_features        = schema.n_features
n_rows_raw        = n_games
n_rows_augmented  = n_games * 2   # after team-swap augmentation
train_rows        = int(n_rows_augmented * 0.80)
val_rows          = n_rows_augmented - train_rows
bytes_per_row     = 9 * 4 * 2    # 9 non-zeros × 4 bytes × (idx + val arrays)
train_mb          = train_rows * bytes_per_row / 1e6
val_mb            = val_rows   * bytes_per_row / 1e6

print(f"\nFeature schema:")
print(f"  Total features : {n_features:,}  "
      f"({n_brawlers} brawlers × 2 teams + {n_maps} maps + {n_modes} modes + 1 skill_ns)")
print(f"  Non-zeros/row  : 9  (compact sparse format)")
print(f"\nFeature matrix (estimated):")
print(f"  Raw rows       : {n_rows_raw:>10,}")
print(f"  After aug.     : {n_rows_augmented:>10,}  (2× after team-swap)")
print(f"  Train rows     : {train_rows:>10,}  (first 80% chronologically)")
print(f"  Val rows       : {val_rows:>10,}  (last 20%)")
print(f"  Train memory   : {train_mb:>7.0f} MB  (compact sparse)")
print(f"  Val memory     : {val_mb:>7.0f} MB")

In [ ]:
# 1.4 — Train FM
#
# Trains the Factorization Machine end-to-end and saves fm_model.pkl.
# Expect this cell to take 5–20 minutes on CPU depending on dataset size.
#
# Progress is printed epoch by epoch. Training stops early when validation
# loss stops improving (controlled by FM_PATIENCE above).

fm = train_fm(
    k            = FM_K,
    lr           = FM_LR,
    weight_decay = FM_WEIGHT_DECAY,
    batch_size   = FM_BATCH_SIZE,
    max_epochs   = FM_MAX_EPOCHS,
    patience     = FM_PATIENCE,
    model_path   = DATA_DIR / "fm_model.pkl",
    schema_path  = SCHEMA_PATH,
    db_path      = DB_PATH,
    elo_min      = AVG_ELO_MIN,
    elo_max      = AVG_ELO_MAX,
)

print(f"\nFM saved → {DATA_DIR / 'fm_model.pkl'}")

In [ ]:
# 1.5 — FM Quality Check
#
# Checks the trained FM against the thresholds required before proceeding to
# self-play. Uses metrics recorded during training — no val set reload needed.
#
# ┌─────────────────┬──────────┬──────────────────────────────────────────────┐
# │ Metric          │ Threshold│ Why it matters                               │
# ├─────────────────┼──────────┼──────────────────────────────────────────────┤
# │ Val log-loss    │ < 0.685  │ Random baseline is 0.693. Any lower means    │
# │                 │          │ the FM learned real signal. Below 0.685 is   │
# │                 │          │ the minimum for trustworthy value labels.     │
# ├─────────────────┼──────────┼──────────────────────────────────────────────┤
# │ Val AUC         │ > 0.550  │ Random is 0.500. A modest AUC (~0.62) is     │
# │                 │          │ normal — Brawl Stars has high draft variance. │
# │                 │          │ Values below 0.550 indicate a data problem.  │
# ├─────────────────┼──────────┼──────────────────────────────────────────────┤
# │ Val Brier score │ < 0.245  │ Random is 0.250. Measures calibration and    │
# │                 │          │ accuracy together. A passing Brier score      │
# │                 │          │ confirms the FM's probabilities are grounded. │
# └─────────────────┴──────────┴──────────────────────────────────────────────┘
#
# If the check fails: try increasing FM_K (e.g. 32 → 64) or reducing
# FM_WEIGHT_DECAY (e.g. 1e-4 → 1e-5), then retrain. If data quality is
# suspect, revisit the ELO filter bounds in cell 1.1.

# Load FM from disk so this cell works even if you skipped training
fm = FMInference.load(DATA_DIR / "fm_model.pkl")

result = check_fm_calibration(fm)

if not result["pass"]:
    print("\n⚠  Do not proceed to Section 2. Retrain the FM or run the "
          "optional hyperparameter sweep in cell 1.6.")
else:
    print("\n✓  FM passed. Proceed to Section 2.")

# Inference speed — directly determines MCTS simulation budget
print()
fm.benchmark(n_calls=20_000)

In [ ]:
# 1.6 — FM Hyperparameter Sweep (OPTIONAL)
#
# Run this cell only if:
#   - The quality check in 1.5 failed, or
#   - You want to squeeze the best possible FM before a long self-play run.
#
# Trains one FM per (k, weight_decay) combination and prints a results table
# sorted by val log-loss. The best configuration is then retrained and saved,
# overwriting fm_model.pkl.
#
# Expected runtime: ~(len(K_GRID) × len(WD_GRID)) × single-train time.
# With defaults below (6 configs), expect 30–90 minutes on CPU.

K_GRID  = [16, 32, 64]      # embedding dimensions to try
WD_GRID = [1e-4, 1e-3]      # weight decay values to try

# ─────────────────────────────────────────────────────────────────────────────
import itertools

sweep_results = []

for k, wd in itertools.product(K_GRID, WD_GRID):
    print(f"\n── Sweep: k={k}  weight_decay={wd} ──")
    _fm = train_fm(
        k            = k,
        lr           = FM_LR,
        weight_decay = wd,
        batch_size   = FM_BATCH_SIZE,
        max_epochs   = FM_MAX_EPOCHS,
        patience     = FM_PATIENCE,
        model_path   = DATA_DIR / f"_sweep_k{k}_wd{wd:.0e}.pkl",
        schema_path  = SCHEMA_PATH,
        db_path      = DB_PATH,
        elo_min      = AVG_ELO_MIN,
        elo_max      = AVG_ELO_MAX,
    )
    sweep_results.append({
        "k": k, "weight_decay": wd,
        "val_logloss": _fm.val_logloss,
        "val_auc":     _fm.val_auc,
        "val_brier":   _fm.val_brier,
    })

# Sort by val log-loss ascending (lower is better)
sweep_results.sort(key=lambda r: r["val_logloss"])

print("\n" + "─" * 64)
print(f"{'k':>4}  {'weight_decay':>12}  {'val_logloss':>11}  {'val_auc':>7}  {'val_brier':>9}")
print("─" * 64)
for r in sweep_results:
    marker = " ← best" if r is sweep_results[0] else ""
    print(f"{r['k']:>4}  {r['weight_decay']:>12.0e}  {r['val_logloss']:>11.4f}  "
          f"{r['val_auc']:>7.4f}  {r['val_brier']:>9.4f}{marker}")
print("─" * 64)

# Retrain with best config and save as the canonical fm_model.pkl
best = sweep_results[0]
print(f"\nRetraining with best config: k={best['k']}  weight_decay={best['weight_decay']:.0e}")
fm = train_fm(
    k            = best["k"],
    lr           = FM_LR,
    weight_decay = best["weight_decay"],
    batch_size   = FM_BATCH_SIZE,
    max_epochs   = FM_MAX_EPOCHS,
    patience     = FM_PATIENCE,
    model_path   = DATA_DIR / "fm_model.pkl",
    schema_path  = SCHEMA_PATH,
    db_path      = DB_PATH,
    elo_min      = AVG_ELO_MIN,
    elo_max      = AVG_ELO_MAX,
)

# Clean up sweep checkpoints
import os
for r in sweep_results:
    p = DATA_DIR / f"_sweep_k{r['k']}_wd{r['weight_decay']:.0e}.pkl"
    if p.exists():
        os.remove(p)

print(f"\nBest FM saved → {DATA_DIR / 'fm_model.pkl'}")
check_fm_calibration(fm)

---
## Section 2 — Matchup Database

**Skip this section** if `matchup_db.pkl` already exists in `DATA_DIR` (shown as `[✓]` in the status check).

---

The Matchup Database is the opponent model used during self-play. When MCTS simulates future picks for the opposing team, it doesn't pick randomly — it prefers brawlers that have historically countered your current team. This is what makes self-play games resemble real adversarial drafting rather than random search.

### Three lookup tables

| Table | Key | What it stores |
|-------|-----|----------------|
| **Brawler stats** | (brawler, map, skill_tier) | Win rate and pick rate per brawler in context |
| **Counter matrix** | (brawler_mine, brawler_opp, map, skill_tier) | P(my team wins \| these two brawlers face off) |
| **Synergy matrix** | (brawler_a, brawler_b, mode, skill_tier) | Win rate delta above per-brawler baselines for same-team pairs |

All three tables share the same **fallback chain**: if a specific (map, tier) cell has too little data, the lookup automatically falls back to (mode, tier) → mode → global. This guarantees that every possible brawler/context combination returns a result.

### Skill tiers

Matches are split into 4 skill tiers using quartiles of the `skill_ns` score. Tier 0 is casual play, tier 3 is elite. The tier boundaries from the training data are stored inside the DB object so that MCTS can map any `skill_ns` value to a tier at runtime without reloading the dataset.

### Why this matters for self-play quality

With a flat random rollout policy, the opponent in simulations makes naive picks. The counter-rate prior concentrates opponent picks on brawlers that actually threaten your current lineup — producing training data that reflects the adversarial structure of ranked drafting. Higher-quality self-play games → better joint network training signal.

In [ ]:
# 2.1 — Build & Validate Matchup Database
#
# Builds all three lookup tables from the season's match data and saves the
# result to matchup_db.pkl. Uses the same DB_PATH and ELO filters as the
# season config — no additional parameters are needed.
#
# Expected runtime: 3–10 minutes (dominated by the counter matrix, which
# processes 9 cross-team slot combinations × the full game dataset).

DB_SAVE_PATH = DATA_DIR / "matchup_db.pkl"

# ELO bounds come from the season config (set in Section 0 cell 0.1).
# Edit these here if you want to override for the matchup DB specifically.
_cfg    = SEASON_CONFIGS[SEASON]
_elo_min = _cfg["elo_min"]
_elo_max = _cfg["elo_max"]

db = MatchupDB.build(
    db_path = DB_PATH,
    elo_min = _elo_min,
    elo_max = _elo_max,
)
db.save(DB_SAVE_PATH)

# ── Coverage stats ────────────────────────────────────────────────────────────
# Load the feature schema to get the canonical brawler vocabulary.
schema = FeatureSchema.load(DATA_DIR / "feature_schema.pkl")
vocab  = schema.vocab

n_brawler_full   = len(db.brawler["full"])
n_brawler_global = len(db.brawler["global"])
n_counter_full   = len(db.counter["full"])
n_counter_global = len(db.counter["global"])
n_synergy_global = len(db.synergy["global"])

q25, q50, q75 = db.skill_tier_boundaries
print(f"\nMatchup DB coverage:")
print(f"  Skill tier boundaries  : Q25={q25:.3f}  Q50={q50:.3f}  Q75={q75:.3f}")
print(f"  Brawler stats (full)   : {n_brawler_full:>8,}  (map × tier cells)")
print(f"  Brawler stats (global) : {n_brawler_global:>8,}  (should equal vocab size: {len(vocab)})")
print(f"  Counter pairs (full)   : {n_counter_full:>8,}  (ordered, map × tier)")
print(f"  Counter pairs (global) : {n_counter_global:>8,}  (ordered, all contexts)")
print(f"  Synergy pairs (global) : {n_synergy_global:>8,}  (canonical, unordered)")

# ── Vocab coverage check ──────────────────────────────────────────────────────
missing_global = [b for b in vocab if (b,) not in db.brawler["global"]]
if missing_global:
    print(f"\n  ⚠  {len(missing_global)} brawlers have no global fallback: {missing_global}")
    print("     These brawlers will not be scored by the rollout opponent model.")
else:
    print(f"\n  ✓  All {len(vocab)} vocabulary brawlers have a global fallback entry.")

# ── Fallback chain sanity check ───────────────────────────────────────────────
# Pick the first brawler in vocab, a common mode, and tier=1. Walk the chain.
test_brawler = vocab[0]
test_mode    = list({k[1] for k in db.brawler["no_tier"]})[0]
test_map     = list({k[1] for k in db.brawler["full"]})[0]

for tier in range(4):
    entry = db.brawler_lookup(test_brawler, test_mode, test_map, tier)
    level_name = ["full (map+tier)", "no_map (mode+tier)", "no_tier (mode)", "global"][entry["level"]]
    print(f"  lookup({test_brawler!r}, tier={tier})  →  "
          f"win_rate={entry['win_rate']:.3f}  pick_rate={entry['pick_rate']:.4f}  "
          f"[{level_name}]")

---
## Section 3 — Self-Play Data Generation

**Skip this section** if `self_play/` already shows a game count in the status check above and you don't want to generate more data.

---

Two MCTS agents — each with its own perspective on the draft board — play complete 6-pick drafts against each other. At every pick slot, the acting agent runs a tree search and returns a visit distribution over available brawlers. That distribution becomes the **policy training target**. After all 6 picks are made, the Factorization Machine evaluates the completed draft and produces a win probability. That value propagates back to all 6 records as the **value training target**.

### Iteration 0 vs. iteration 1+

**Iteration 0** has no learned prior. MCTS explores using:
- UCB1 selection (no PUCT, no policy head)
- FM evaluations at terminal nodes via rollouts

**Iteration 1+** uses the best joint network trained so far:
- PUCT selection: the policy head provides an informed prior over actions
- The value head evaluates leaf nodes directly, skipping rollouts entirely

The value head is much cheaper than a rollout (one forward pass vs. a random simulation to a terminal state), so the simulation budget can be increased in later iterations without proportionally increasing wall-clock time.

### The training loop

Each iteration follows the same three-step cycle:

```
generate games  →  train joint net  →  evaluate (promote if better)
```

- **Cell 3.4** runs one iteration (game generation only). Re-run it to advance.
- **Section 5** wraps cells 3.4 → 4.5 into a single automated loop.

### Section independence

All cells in this section reload their dependencies from disk (`fm_model.pkl`,
`matchup_db.pkl`, `joint_net.pkl`). Run cell 3.1 first to set parameters,
then run any other cell independently.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  3.1 — SELF-PLAY PARAMETERS                                 ║
# ║  Tune these before running cells 3.2–3.4.                   ║
# ╚══════════════════════════════════════════════════════════════╝

# --- Game Generation ---
N_GAMES_PER_ITER = 500
# Games to generate per self-play iteration. Each game produces 6 training
# records (one per pick slot). More games = better training data, slower
# iteration. Good range: 200–500. Reduce to 100 for a quick sanity check.

N_WORKERS = 4
# Parallel worker processes. Set to your CPU core count minus 1.
# Use 1 to disable multiprocessing (simpler output, easier to debug).
# Workers load fm_model.pkl and matchup_db.pkl independently at startup.

BAN_PROBABILITY = 0.35
# Per-brawler probability that each top-pick-rate brawler gets banned.
# At 0.35, dominant meta brawlers are absent from ~35% of games, forcing
# the policy to learn non-trivial picks. Range: 0.1–0.5.

MAX_BANS = 6
# Hard cap on total bans per game. With BAN_PROBABILITY=0.35 over the top-15
# brawlers, expected bans ≈ 5.3 before the cap — so MAX_BANS only activates
# in the tail. Increase to allow more bans if desired.

# --- MCTS Simulation Budget ---
N_SIMS_ITER0 = 2000
# Simulations per pick in iteration 0 (no learned prior).
# Each sim runs a full rollout to a terminal state using FM evaluations.
# Reduce to 1000 if iteration 0 is too slow on your hardware.

N_SIMS_ITER_N = 10000
# Simulations per pick in iteration 1+ (with joint net as prior + value).
# The value head replaces rollouts, making each sim much cheaper than iter 0.
# Can be set higher than N_SIMS_ITER0 without a proportional time increase.
# Reduce to 2000–3000 if still slow after accounting for the value head speedup.

# --- MCTS Tree Parameters ---
UCB1_C = 0.5
# Exploration constant used in UCB1 (iteration 0) and PUCT (iteration 1+).
# Lower than sqrt(2) ≈ 1.41 because the draft tree is only 6 picks deep —
# less depth means less need for broad exploration. Range: 0.3–1.5.

PUCT_ALPHA = 0.7
# Weight of the policy head prior in the PUCT selection formula.
# Higher = stronger influence from the learned prior; lower = more UCB1-like.
# Range: 0.3–0.9. Only takes effect in iteration 1+ when a joint net exists.

MIN_PICK_RATE = 0.003
# Brawlers below this global pick rate on the game's map/mode/tier are
# excluded from the MCTS tree (the action space). Reduces branching factor
# from ~100 brawlers to ~40–50, cutting node expansion cost roughly in half.
# Does not affect the rollout policy, which always uses the full vocab.

# --- Training Loop (used by Section 5 automated loop) ---
N_ITERS = 2
# Total self-play → train → evaluate iterations to run in the Section 5 loop.
# Each iteration generates N_GAMES_PER_ITER games and trains one joint net.
# You can run fewer iterations manually via cell 3.4 + Section 4.

PROMOTION_WIN_THR = 0.515
# The new joint net must achieve at least this FM win rate against the current
# best net (in N_EVAL_GAMES head-to-head games) to be promoted to joint_net.pkl.
# A threshold slightly above 0.5 filters out noise without requiring dominance.

N_EVAL_GAMES = 200
# Number of head-to-head games used to evaluate whether to promote a new net.
# More games = lower variance estimate, but slower per iteration.

# ─────────────────────────────────────────────────────────────────────────────
print("Self-play parameters set:")
print(f"  N_GAMES_PER_ITER : {N_GAMES_PER_ITER:,}")
print(f"  N_WORKERS        : {N_WORKERS}")
print(f"  N_SIMS_ITER0     : {N_SIMS_ITER0:,}  (iteration 0, no prior)")
print(f"  N_SIMS_ITER_N    : {N_SIMS_ITER_N:,}  (iteration 1+, with joint net)")
print(f"  UCB1_C           : {UCB1_C}")
print(f"  PUCT_ALPHA       : {PUCT_ALPHA}")
print(f"  MIN_PICK_RATE    : {MIN_PICK_RATE}")
print(f"  BAN_PROBABILITY  : {BAN_PROBABILITY}  (per top-brawler)")
print(f"  MAX_BANS         : {MAX_BANS}")
print(f"  N_ITERS          : {N_ITERS}  (Section 5 automated loop)")
print(f"  PROMOTION_WIN_THR: {PROMOTION_WIN_THR}")
print(f"  N_EVAL_GAMES     : {N_EVAL_GAMES}")

In [ ]:
# 3.2 — Speed Benchmark (Run Once)
#
# Run this cell BEFORE generating data for the first time to calibrate your
# simulation budget. It measures iteration-0 MCTS speed (no learned prior,
# pure FM rollouts) at four sim counts and extrapolates total wall-clock time
# for your configured N_GAMES_PER_ITER × N_ITERS.
#
# Based on the output, adjust N_SIMS_ITER0 and N_GAMES_PER_ITER in cell 3.1
# before committing to a full run.
#
# This cell reloads fm_model.pkl and matchup_db.pkl from disk directly.
# It runs correctly whether or not Sections 1–2 were executed this session.

from workflow_utils import benchmark_mcts_speed

_map_mode_pairs = load_map_mode_pairs(
    DB_PATH,
    known_maps=set(FeatureSchema.load(DATA_DIR / "feature_schema.pkl").maps),
)

benchmark_mcts_speed(
    data_dir         = DATA_DIR,
    map_mode_pairs   = _map_mode_pairs,
    sim_counts       = [500, 1000, 2000, 5000],
    n_test_games     = 3,
    n_games_per_iter = N_GAMES_PER_ITER,
    n_iters          = N_ITERS,
    n_workers        = N_WORKERS,
)

In [ ]:
# 3.3 — Replay Buffer Status
#
# Reads the replay buffer manifest from DATA_DIR/self_play/ and prints a
# summary of what's already been generated. If the buffer is empty, this
# is iteration 0.
#
# This cell reads from disk only and runs independently of cells 3.1–3.2.

import json as _json
from collections import Counter as _Counter

_sp_dir       = DATA_DIR / "self_play"
_manifest_path = _sp_dir / "manifest.json"

if not _manifest_path.exists():
    print("Replay buffer is empty — no manifest found.")
    print(f"Path checked: {_manifest_path}")
    print("\nThis is iteration 0. Run cell 3.4 to generate the first batch of games.")
else:
    with open(_manifest_path) as _fh:
        _manifest = _json.load(_fh)

    _batches        = _manifest.get("batches", [])
    _total_records  = sum(b.get("n_records", 0) for b in _batches)
    _total_games    = _total_records // 6
    _n_batches      = len(_batches)
    _next_iter      = _n_batches   # next iteration index = number of batches run so far

    print(f"Replay buffer: {_sp_dir}")
    print(f"  Batches (iterations run) : {_n_batches}")
    print(f"  Total games (all time)   : {_total_games:,}")
    print(f"  Total records (all time) : {_total_records:,}  ({_total_games:,} games × 6 picks)")
    print(f"  Next iteration index     : {_next_iter}")

    # Per-batch breakdown
    if _n_batches > 0:
        print(f"\n  Per-batch breakdown:")
        print(f"  {'Iter':>5}  {'Records':>8}  {'Games':>6}  File")
        for _i, _b in enumerate(_batches):
            _g = _b.get("n_records", 0) // 6
            _fname = Path(_b.get("path", "")).name
            print(f"  {_i:>5}  {_b.get('n_records', 0):>8,}  {_g:>6,}  {_fname}")

    # Records by pick depth — load active buffer (up to max_games most recent games)
    _buf     = ReplayBuffer.load(_sp_dir)
    _records = _buf.all_records()

    if _records:
        _by_depth = _Counter(r.pick_number for r in _records)
        _min_depth_count = min(_by_depth.get(d, 0) for d in range(6))
        _depth_ok = all(_by_depth.get(d, 0) >= 100 for d in range(6))

        print(f"\n  Active buffer: {_buf.n_games:,} games  ({_buf.n_records:,} records)")
        print(f"  Records by pick depth:")
        for _d in range(6):
            _cnt = _by_depth.get(_d, 0)
            _flag = "" if _cnt >= 100 else "  ← sparse (<100)"
            print(f"    depth {_d}: {_cnt:>6,}{_flag}")

        _coverage_status = "OK" if _depth_ok else "SPARSE — may need more games before training"
        print(f"\n  Coverage: {_coverage_status}")
    else:
        print("\n  Active buffer is empty.")

In [ ]:
# 3.4 — Run One Self-Play Iteration
#
# Generates N_GAMES_PER_ITER self-play games and appends them to the replay
# buffer on disk. Each re-run of this cell advances the iteration counter by
# one. The manifest tracks iteration number — no global state is needed.
#
# Iteration 0: pure FM rollout MCTS (no joint net). Uses N_SIMS_ITER0.
# Iteration 1+: joint_net.pkl serves as policy prior + rollout-free value
#               estimator. Uses N_SIMS_ITER_N (can be higher; each sim is
#               cheaper because the value head skips the rollout phase).
#
# Re-run this cell as many times as desired to accumulate more training data.
# Each run is idempotent: it appends a new batch and updates the manifest.
#
# This cell reloads all dependencies from disk and runs independently of any
# other section. Section 0 (cell 0.1) must have been run to set DATA_DIR,
# DB_PATH, and the parameters from cell 3.1 must be set.

import json as _json
import time as _time

# ── Determine current iteration from manifest ─────────────────────────────────
_manifest_path = DATA_DIR / "self_play" / "manifest.json"
_iter_num = 0
_start_game_idx = 0

if _manifest_path.exists():
    with open(_manifest_path) as _fh:
        _cur_manifest = _json.load(_fh)
    _batches_so_far = _cur_manifest.get("batches", [])
    _iter_num       = len(_batches_so_far)
    # Total games ever generated (including evicted) — used to keep game_id
    # globally unique and P1/P2 alternation correct across batch calls.
    _start_game_idx = sum(b.get("n_records", 0) for b in _batches_so_far) // 6

print(f"Starting self-play iteration {_iter_num}")
print(f"  start_game_idx : {_start_game_idx:,}  (globally unique offset for this batch)")

# ── Decide simulation budget and whether to use joint net ────────────────────
_joint_path = DATA_DIR / "joint_net.pkl"
if _joint_path.exists():
    _n_sims = N_SIMS_ITER_N
    print(f"  joint_net.pkl  : found — using as policy prior + rollout-free value head")
    print(f"  n_sims/pick    : {_n_sims:,}  (N_SIMS_ITER_N, rollout-free mode)")
else:
    _n_sims = N_SIMS_ITER0
    print(f"  joint_net.pkl  : not found — iteration 0 mode (pure FM rollouts)")
    print(f"  n_sims/pick    : {_n_sims:,}  (N_SIMS_ITER0)")

# ── Load map/mode pairs ───────────────────────────────────────────────────────
_map_mode_pairs = load_map_mode_pairs(
    DB_PATH,
    known_maps=set(FeatureSchema.load(DATA_DIR / "feature_schema.pkl").maps),
)
print(f"  map/mode pairs : {len(_map_mode_pairs)}  unique (map, mode) contexts")
print(f"  n_games        : {N_GAMES_PER_ITER:,}")
print(f"  n_workers      : {N_WORKERS}")
print()

# ── Load existing replay buffer from disk ────────────────────────────────────
_buf = ReplayBuffer.load(DATA_DIR / "self_play")
_games_before = _buf.n_games

# ── Run self-play batch ───────────────────────────────────────────────────────
_t0 = _time.perf_counter()

run_self_play_batch(
    n_games         = N_GAMES_PER_ITER,
    data_dir        = DATA_DIR,
    map_mode_pairs  = _map_mode_pairs,
    replay_buffer   = _buf,
    n_workers       = N_WORKERS,
    start_game_idx  = _start_game_idx,
    seed            = _iter_num,      # deterministic per iteration; change to None for random
    n_sims_per_pick = _n_sims,
    min_pick_rate   = MIN_PICK_RATE,
    ban_p           = BAN_PROBABILITY,
    max_bans        = MAX_BANS,
)

_elapsed      = _time.perf_counter() - _t0
_games_added  = _buf.n_games - _games_before
_secs_per_game = _elapsed / max(_games_added, 1)
_games_per_min = 60.0 / _secs_per_game if _secs_per_game > 0 else 0.0

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\nIteration {_iter_num} complete:")
print(f"  Games generated  : {_games_added:,}")
print(f"  Records generated: {_games_added * 6:,}")
print(f"  Elapsed          : {_elapsed:.1f}s")
print(f"  Speed            : {_games_per_min:.1f} games/min  ({_secs_per_game:.2f}s/game)")
print(f"  Buffer total     : {_buf.n_games:,} games  ({_buf.n_records:,} records)")
print(f"\n▶  Next: run Section 4 cells to train the joint network on this data.")

---
## Section 4 — Joint Policy+Value Network

**Skip this section** if `joint_net.pkl` already exists and you don't want to retrain. To add more training data and retrain, run Section 3 first, then come back here.

---

The joint network is a single MLP with a **shared trunk** that feeds two output heads:

| Head | Output | Trained against | Used during MCTS |
|------|--------|-----------------|------------------|
| **Policy** | Probability over brawlers | MCTS visit distributions from self-play | PUCT prior at expansion — concentrates simulations on promising picks |
| **Value** | Win probability ∈ (0, 1) | FM terminal evaluations from self-play | Leaf evaluation — replaces random rollouts entirely |

Training augments the replay buffer with team-swapped copies. Swapped records contribute only to the value head (the policy target is asymmetric and invalid after swapping).

### Early stopping

Training stops when the combined validation loss (`policy_CE + λ_v × value_BCE`) stops improving for `JOINT_PATIENCE` consecutive epochs. The combined loss is noisier than the FM's log-loss, so a higher patience (7 vs. 5) is used.

### Promotion

After training, the new network plays `N_EVAL_GAMES` head-to-head games against the current best `joint_net.pkl` (or against FM-only MCTS in the first iteration). If the new network wins ≥ `PROMOTION_WIN_THR` of games, it is saved as the new `joint_net.pkl`.

### Section independence

Cells 4.2–4.5 reload their dependencies from disk where possible. The within-section dependency chain is:
`4.1 (params) → 4.2 (load data) → 4.3 (train) → 4.4 (validate) → 4.5 (promote)`

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  4.1 — JOINT NETWORK HYPERPARAMETERS                        ║
# ║  Tune these before running cells 4.2–4.5.                   ║
# ╚══════════════════════════════════════════════════════════════╝

# --- Architecture ---
JOINT_HIDDEN_DIMS = [256, 128]
# Hidden layer sizes for the shared trunk (passed to JointNet as trunk_hidden).
# These control model capacity — wider/deeper = more expressive but needs more data.
# Recommended by replay buffer size:
#   < 2k games  →  [128, 64]    (default for early iterations)
#   2k–10k games →  [256, 128]  (good default)
#   > 10k games  →  [512, 256]  (matches the module default; use for late-stage training)
# The architecture sweep in cell 4.6 searches this space automatically.

# --- Training ---
JOINT_LR = 1e-3
# Adam learning rate. The default works well; reduce to 3e-4 if combined val
# loss oscillates without decreasing.

JOINT_WEIGHT_DECAY = 1e-4
# L2 regularisation. Increase to 1e-3 if val loss tracks well above train loss.

JOINT_BATCH_SIZE = 512
# Mini-batch size. Increase to 1024 for large buffers (> 5k games) to speed
# up training. Reduce to 256 if you run out of memory.

JOINT_MAX_EPOCHS = 50
# Hard training cap. Early stopping (JOINT_PATIENCE) triggers before this
# in almost all cases.

JOINT_PATIENCE = 7
# Epochs without combined val loss improvement before stopping. Slightly
# higher than the FM (5) because the joint loss is noisier due to the two-head
# structure and the policy cross-entropy's dependence on visit distribution quality.

# --- Loss Weighting ---
POLICY_LOSS_WEIGHT = 1.0
# Multiplier on policy cross-entropy. Increasing this improves the policy head
# but can hurt value head calibration.

VALUE_LOSS_WEIGHT = 1.0
# Multiplier on value BCE (λ_v in the combined loss). Increasing this improves
# value head calibration but slightly reduces policy head accuracy.
# Increase to 2.0 if cell 4.4's value calibration check fails.

# --- Data ---
JOINT_VAL_FRAC = 0.1
# Fraction of games (by game_id) held out for validation.
# The train/val split is always chronological (oldest games → train).
# 0.1 = 10% val; increase to 0.2 for larger buffers where more val data is available.

# --- Promotion (used by cell 4.5) ---
PROMOTION_WIN_THR = 0.525
# New net must win at least this fraction of evaluation games to be promoted
# to joint_net.pkl. Slightly above 0.5 to avoid promoting on noise. Lower
# this threshold if your hardware doesn't allow enough eval games to be confident.

N_EVAL_GAMES = 200
# Head-to-head games to run for promotion evaluation. More = more reliable
# estimate, but slower. 200 games gives a standard error of ~0.035 on win rate.

N_EVAL_SIMS = 500
# Simulation budget per pick in evaluation games. Lower than training (2000–5000)
# because eval games only need enough sims to differentiate policy quality —
# they don't need training-quality visit distributions.

# These match the Section 3 defaults. Edit them here if running Section 4 standalone.
EVAL_MIN_PICK_RATE = 0.005  # MCTS tree vocab filter (mirrors MIN_PICK_RATE in 3.1)
EVAL_PUCT_ALPHA    = 0.7    # PUCT prior weight (mirrors PUCT_ALPHA in 3.1)

# ─────────────────────────────────────────────────────────────────────────────
print("Joint network hyperparameters set:")
print(f"  JOINT_HIDDEN_DIMS  : {JOINT_HIDDEN_DIMS}")
print(f"  JOINT_LR           : {JOINT_LR}")
print(f"  JOINT_WEIGHT_DECAY : {JOINT_WEIGHT_DECAY}")
print(f"  JOINT_BATCH_SIZE   : {JOINT_BATCH_SIZE}")
print(f"  JOINT_MAX_EPOCHS   : {JOINT_MAX_EPOCHS}  (patience={JOINT_PATIENCE})")
print(f"  POLICY_LOSS_WEIGHT : {POLICY_LOSS_WEIGHT}")
print(f"  VALUE_LOSS_WEIGHT  : {VALUE_LOSS_WEIGHT}  (λ_v)")
print(f"  JOINT_VAL_FRAC     : {JOINT_VAL_FRAC}")
print(f"\nPromotion settings:")
print(f"  PROMOTION_WIN_THR  : {PROMOTION_WIN_THR}")
print(f"  N_EVAL_GAMES       : {N_EVAL_GAMES}")
print(f"  N_EVAL_SIMS        : {N_EVAL_SIMS}  sims/pick in eval games")

In [ ]:
# 4.2 — Prepare Training Data
#
# Loads the replay buffer from disk and shows the data available for training.
# Run this cell before cell 4.3 to confirm coverage is adequate.
#
# train_joint() handles team-swap augmentation internally, so this cell only
# inspects the original records. The augmentation doubles the value-head
# training set: every original record gets a swapped copy where my_team ↔
# opp_team and terminal_win_prob → 1 - terminal_win_prob.
#
# This cell is section-independent: it reads only from DATA_DIR/self_play/,
# which is set in Section 0.

from collections import Counter as _Counter
from self_play import verify_buffer_coverage

_sp_dir = DATA_DIR / "self_play"

assert _sp_dir.exists(), (
    f"No self-play directory at {_sp_dir}\n"
    "Run cells 3.1 and 3.4 to generate games before training."
)

# Load the active replay buffer (up to max_games most recent games).
_buf = ReplayBuffer.load(_sp_dir)
assert _buf.n_games > 0, (
    f"Replay buffer is empty at {_sp_dir}\n"
    "Run cell 3.4 to generate games."
)

records = _buf.all_records()   # flat list — kept in session for cells 4.3–4.4

# ── Summary ───────────────────────────────────────────────────────────────────
_n_orig       = len(records)
_n_with_dist  = sum(1 for r in records if r.visit_dist)
_by_depth     = _Counter(r.pick_number for r in records)
_win_prob_mean = sum(r.terminal_win_prob for r in records) / _n_orig

print(f"Replay buffer: {_sp_dir}")
print(f"  Games in active buffer  : {_buf.n_games:,}")
print(f"  Records (original)      : {_n_orig:,}")
print(f"  Records with visit_dist : {_n_with_dist:,}  (contribute to policy head)")
print(f"  Records without dist    : {_n_orig - _n_with_dist:,}  (value head only)")
print(f"  Augmented (estimated)   : +{_n_orig:,}  → {_n_orig * 2:,} total for value training")
print(f"  Mean terminal_win_prob  : {_win_prob_mean:.4f}  (expected ~0.50)")

# Val split preview (mirrors train_joint's internal split)
_all_game_ids = sorted({r.game_id for r in records})
_n_val_games  = max(1, int(len(_all_game_ids) * JOINT_VAL_FRAC))
_val_ids      = set(_all_game_ids[-_n_val_games:])
_n_val_rec    = sum(1 for r in records if r.game_id in _val_ids)
print(f"\n  Train/val split (JOINT_VAL_FRAC={JOINT_VAL_FRAC}):")
print(f"    Train games  : {len(_all_game_ids) - _n_val_games:,}  ({_n_orig - _n_val_rec:,} records)")
print(f"    Val games    : {_n_val_games:,}  ({_n_val_rec:,} records)")

# Pick-depth breakdown
print(f"\n  Records by pick depth:")
for _d in range(6):
    _cnt  = _by_depth.get(_d, 0)
    _flag = "" if _cnt >= 100 else "  ← sparse (<100)"
    print(f"    depth {_d}: {_cnt:,}{_flag}")

# Coverage gate: train_joint is safe to run only if all depths have ≥ 100 records.
print()
_cov = verify_buffer_coverage(records)

if not _cov["pass"]:
    print("\n⚠  Coverage check failed. Generate more games (cell 3.4) before training.")
    print("   Each iteration of cell 3.4 adds 6 records per game per pick depth.")
else:
    print("\n✓  Coverage OK. Proceed to cell 4.3.")

In [ ]:
# 4.3 — Train Joint Network
#
# Trains the JointNet on the replay buffer loaded in cell 4.2.
# Prints epoch-by-epoch: combined loss, policy component, value component,
# and val combined loss. Saves a versioned checkpoint after training.
#
# Prerequisite: run cell 4.2 first to load `records` and check coverage.
# This cell depends on records (from 4.2) and all parameters from 4.1.

import json as _json

# ── Determine current iteration from manifest (for versioned filename) ────────
_manifest_path = DATA_DIR / "self_play" / "manifest.json"
_iter_num = 0
if _manifest_path.exists():
    with open(_manifest_path) as _fh:
        _m = _json.load(_fh)
    _iter_num = len(_m.get("batches", []))

print(f"Training joint network on {len(records):,} records  "
      f"(iteration {_iter_num}  |  arch={JOINT_HIDDEN_DIMS}  |  device={DEVICE})")
print()

# ── Load feature schema ───────────────────────────────────────────────────────
_schema = FeatureSchema.load(DATA_DIR / "feature_schema.pkl")

# ── Train ─────────────────────────────────────────────────────────────────────
# loss_history captures (epoch, policy_component, value_component, val_combined)
# tuples for use in cell 4.4's calibration check.
_loss_history = []

_new_joint = train_joint(
    records           = records,
    schema            = _schema,
    lambda_v          = VALUE_LOSS_WEIGHT,
    lr                = JOINT_LR,
    weight_decay      = JOINT_WEIGHT_DECAY,
    batch_size        = JOINT_BATCH_SIZE,
    max_epochs        = JOINT_MAX_EPOCHS,
    patience          = JOINT_PATIENCE,
    val_game_fraction = JOINT_VAL_FRAC,
    trunk_hidden      = tuple(JOINT_HIDDEN_DIMS),
    device            = DEVICE,
    verbose           = True,
    loss_history      = _loss_history,
)

# ── Save versioned checkpoint ─────────────────────────────────────────────────
# joint_net_iter_N.pkl is a permanent record of each iteration's trained net.
# joint_net.pkl (the "best") is written only after promotion in cell 4.5.
_versioned_path = DATA_DIR / f"joint_net_iter_{_iter_num}.pkl"
_new_joint.save(_versioned_path)

# ── Loss summary ──────────────────────────────────────────────────────────────
if _loss_history:
    _best_epoch = min(_loss_history, key=lambda t: t[3])   # (epoch, pol, val_head, val_combined)
    _last_epoch = _loss_history[-1]
    print(f"\nTraining summary:")
    print(f"  Epochs run     : {_last_epoch[0] + 1}")
    print(f"  Best val epoch : {_best_epoch[0]}  (val_combined={_best_epoch[3]:.4f})")
    print(f"  Final train    : policy={_last_epoch[1]:.4f}  value={_last_epoch[2]:.4f}")

print(f"\n✓  Versioned checkpoint: {_versioned_path.name}")
print(f"   Run cell 4.4 to validate, then cell 4.5 for promotion evaluation.")

In [ ]:
# 4.4 — Joint Network Validation
#
# Two quick checks before promotion evaluation:
#
#   1. Value head calibration — bins predicted win probabilities vs. FM labels.
#      Pass threshold: mean absolute deviation < 0.05 across 10 equal-width bins.
#      A perfectly calibrated model has 0.0 deviation; random guessing has ~0.25.
#
#   2. Policy head sanity — top-1 agreement between the network's policy head
#      and the MCTS visit distribution that generated each state. Expect ≥ 3/5
#      on a random sample of 5 records. Lower agreement is expected in early
#      iterations (iteration 0 data has no policy prior) and improves over time.
#
# Prerequisite: run cells 4.2 (records) and 4.3 (_new_joint) first.

import numpy as _np

# ── Replicate train_joint's val split ─────────────────────────────────────────
# Evaluate calibration on held-out val records to avoid over-optimistic results.
_all_ids     = sorted({r.game_id for r in records})
_n_val_games = max(1, int(len(_all_ids) * JOINT_VAL_FRAC))
_val_ids     = set(_all_ids[-_n_val_games:])
_val_records = [r for r in records if r.game_id in _val_ids]

print(f"Validation set: {len(_val_records):,} records from {_n_val_games:,} held-out games")
print()

# ═══════════════════════════════════════════════════════════════════════════════
# Check 1 — Value head calibration
# ═══════════════════════════════════════════════════════════════════════════════
_pred_probs = _np.array([_new_joint.evaluate(r.state) for r in _val_records])
_true_probs = _np.array([r.terminal_win_prob for r in _val_records])

_n_bins = 10
_bin_edges = _np.linspace(0.0, 1.0, _n_bins + 1)
_bin_mads  = []   # mean absolute deviation per occupied bin

print("Value head calibration (10 equal-width bins):")
print(f"  {'Pred range':>14}  {'N':>6}  {'Mean pred':>10}  {'Mean FM':>8}  {'|diff|':>7}")
print("  " + "─" * 54)

for _lo, _hi in zip(_bin_edges[:-1], _bin_edges[1:]):
    _mask = (_pred_probs >= _lo) & (_pred_probs < _hi)
    if _mask.sum() == 0:
        continue
    _mp    = float(_pred_probs[_mask].mean())
    _mt    = float(_true_probs[_mask].mean())
    _diff  = abs(_mp - _mt)
    _bin_mads.append(_diff)
    _flag = "  ←" if _diff > 0.05 else ""
    print(f"  [{_lo:.1f}, {_hi:.1f})      {_mask.sum():>6,}  {_mp:>10.4f}  {_mt:>8.4f}  {_diff:>7.4f}{_flag}")

_val_cal_mae = float(_np.mean(_bin_mads)) if _bin_mads else 1.0
_cal_pass = _val_cal_mae < 0.05

print(f"\n  Mean calibration error : {_val_cal_mae:.4f}  (threshold < 0.05)")
print(f"  Value calibration      : {'PASS ✓' if _cal_pass else 'FAIL ✗'}")

if not _cal_pass:
    print("  → If calibration is poor: increase VALUE_LOSS_WEIGHT (e.g. 1.0 → 2.0) and retrain.")
    print("    Also verify the FM quality check passed in cell 1.5 — bad FM labels = bad value targets.")

print()

# ═══════════════════════════════════════════════════════════════════════════════
# Check 2 — Policy head sanity (top-1 agreement with MCTS visit distributions)
# ═══════════════════════════════════════════════════════════════════════════════
# Sample 5 records with non-empty visit_dist from the val set.
_policy_records = [r for r in _val_records if r.visit_dist]

if len(_policy_records) < 5:
    print("Policy sanity: not enough val records with visit_dist for check (need ≥ 5).")
    _pol_pass = True   # don't block on this — might happen in very early iterations
else:
    _rng = _np.random.default_rng(0)
    _sample_idx = _rng.choice(len(_policy_records), size=5, replace=False)
    _sample = [_policy_records[i] for i in _sample_idx]

    print("Policy head sanity check (5 sampled val records):")
    print(f"  {'Depth':>5}  {'Joint top-1':>16}  {'MCTS top-1':>16}  {'Match':>5}")
    print("  " + "─" * 50)

    _n_agree = 0
    for _r in _sample:
        _used      = _r.state.my_team | _r.state.opp_team | (_r.state.bans or frozenset())
        _available = [b for b in _new_joint.schema.vocab if b not in _used]
        _priors    = _new_joint.predict_prior(_r.state, _available)
        _joint_top = _available[int(_np.argmax(_priors))]
        _mcts_top  = max(_r.visit_dist, key=_r.visit_dist.get)
        _match     = (_joint_top == _mcts_top)
        _n_agree  += int(_match)
        print(f"  {_r.pick_number:>5}  {_joint_top:>16}  {_mcts_top:>16}  {'✓' if _match else '✗':>5}")

    _pol_pass = _n_agree >= 3
    print(f"\n  Top-1 agreement : {_n_agree}/5  (threshold ≥ 3/5)")
    print(f"  Policy sanity   : {'PASS ✓' if _pol_pass else 'FAIL ✗'}")

    if not _pol_pass:
        print("  → Low agreement is normal in iteration 0 (visit dists are coarse).")
        print("    It should improve in later iterations as the policy prior guides search.")

# ── Overall verdict ───────────────────────────────────────────────────────────
print()
if _cal_pass and _pol_pass:
    print("All checks passed ✓  Proceed to cell 4.5 for promotion evaluation.")
else:
    print("One or more checks failed. See guidance above.")
    print("You may still run cell 4.5, but consider retraining with adjusted hyperparameters.")

In [ ]:
# 4.5 — Promotion Evaluation
#
# Runs N_EVAL_GAMES head-to-head games: _new_joint (from cell 4.3) vs. the
# current best joint_net.pkl on disk (or FM-only MCTS if none exists yet).
#
# Each game assigns _new_joint to one team and the baseline to the other.
# P1/P2 roles alternate by game index so neither net benefits from picking
# first more than half the time. Both nets run their own independent MCTS
# trees using their own policy + value head at each pick slot.
#
# After all games, the mean FM-evaluated terminal win probability for
# _new_joint's team is reported. If it meets PROMOTION_WIN_THR, _new_joint
# is saved as joint_net.pkl (the new "current best").
#
# Prerequisite: run cell 4.3 to produce _new_joint.

import time as _time
from workflow_utils import run_eval_games

# ── Load baseline ─────────────────────────────────────────────────────────────
_joint_path      = DATA_DIR / "joint_net.pkl"
_baseline_joint  = None

if _joint_path.exists():
    _baseline_joint = JointNetInference.load(_joint_path)
    print(f"Baseline  : joint_net.pkl  (current best, loaded from disk)")
else:
    print(f"Baseline  : FM-only MCTS  (no joint_net.pkl — this is the first promotion)")

print(f"Challenger: _new_joint  (trained in cell 4.3)")
print(f"Games     : {N_EVAL_GAMES}  ({N_EVAL_SIMS} sims/pick, serial)")
print()

# ── Run evaluation ────────────────────────────────────────────────────────────
_eval_result = run_eval_games(
    new_joint      = _new_joint,
    baseline_joint = _baseline_joint,
    data_dir       = DATA_DIR,
    db_path        = DB_PATH,
    n_games        = N_EVAL_GAMES,
    n_sims         = N_EVAL_SIMS,
    min_pick_rate  = EVAL_MIN_PICK_RATE,
    puct_alpha     = EVAL_PUCT_ALPHA,
    seed           = 42,
)

# ── Results ───────────────────────────────────────────────────────────────────
_win_rate  = _eval_result["new_net_win_rate"]
_elapsed   = _eval_result["elapsed_sec"]
_promote   = _win_rate >= PROMOTION_WIN_THR

import numpy as _np
_win_probs_arr = _np.array(_eval_result["win_probs"])

print(f"Evaluation results:")
print(f"  Games played     : {N_EVAL_GAMES}")
print(f"  Elapsed          : {_elapsed:.1f}s  ({_elapsed / N_EVAL_GAMES:.2f}s/game)")
print(f"  New net win rate : {_win_rate:.4f}  (mean FM terminal win prob)")
print(f"  Std deviation    : {_win_probs_arr.std():.4f}")
print(f"  Promotion thresh : {PROMOTION_WIN_THR}")
print(f"  Decision         : {'PROMOTE ✓' if _promote else 'REJECT ✗'}")
print()

if _promote:
    _new_joint.save(DATA_DIR / "joint_net.pkl")
    print(f"✓  PROMOTED: new net saved as joint_net.pkl")
    print(f"   It will be used as the policy prior + value estimator in future self-play iterations.")
    print(f"   Re-run cell 3.4 → 4.3 → 4.5 to continue training.")
else:
    print(f"✗  NOT PROMOTED  (win rate {_win_rate:.4f} < {PROMOTION_WIN_THR})")
    print(f"   Options:")
    print(f"   • Generate more self-play games (cell 3.4) and retrain")
    print(f"   • Lower PROMOTION_WIN_THR in cell 4.1 (accept smaller improvements)")
    print(f"   • Increase N_EVAL_SIMS for a more accurate win rate estimate")
    print(f"   • Run the architecture sweep (cell 4.6) to find a better model config")

In [ ]:
# 4.6 — Architecture Sweep (OPTIONAL)
#
# Run this cell only if:
#   - Cell 4.4's value calibration check failed, or
#   - You have accumulated a large replay buffer (> 5k games) and want to
#     use a more expressive architecture, or
#   - You want to find the best VALUE_LOSS_WEIGHT for your data.
#
# Trains one joint net per (hidden_dims, value_loss_weight) combination and
# prints a table sorted by val combined loss. The best configuration is then
# retrained and stored as _new_joint, ready for cell 4.5.
#
# Prerequisite: run cell 4.2 first to load `records`. No other cells needed.
# Expected runtime: ~(len(ARCH_GRID) × len(WD_GRID)) × single-train time.

import itertools as _itertools

ARCH_GRID   = [[128, 64], [256, 128], [512, 256]]   # trunk architectures to try
VLW_GRID    = [0.5, 1.0, 2.0]                        # VALUE_LOSS_WEIGHT values to try

# ─────────────────────────────────────────────────────────────────────────────
_schema = FeatureSchema.load(DATA_DIR / "feature_schema.pkl")
_sweep_results = []

for _dims, _vlw in _itertools.product(ARCH_GRID, VLW_GRID):
    print(f"\n── Sweep: arch={_dims}  value_loss_weight={_vlw} ──")
    _lh = []
    _net = train_joint(
        records           = records,
        schema            = _schema,
        lambda_v          = _vlw,
        lr                = JOINT_LR,
        weight_decay      = JOINT_WEIGHT_DECAY,
        batch_size        = JOINT_BATCH_SIZE,
        max_epochs        = JOINT_MAX_EPOCHS,
        patience          = JOINT_PATIENCE,
        val_game_fraction = JOINT_VAL_FRAC,
        trunk_hidden      = tuple(_dims),
        device            = DEVICE,
        verbose           = False,   # suppress per-epoch output during sweep
        loss_history      = _lh,
    )
    _best_val = min(t[3] for t in _lh) if _lh else float("inf")

    # Value calibration error for this config
    _all_ids   = sorted({r.game_id for r in records})
    _n_val     = max(1, int(len(_all_ids) * JOINT_VAL_FRAC))
    _val_ids_s = set(_all_ids[-_n_val:])
    _val_recs  = [r for r in records if r.game_id in _val_ids_s]

    import numpy as _np2
    _preds = _np2.array([_net.evaluate(r.state) for r in _val_recs])
    _trues = _np2.array([r.terminal_win_prob     for r in _val_recs])
    _be    = _np2.linspace(0.0, 1.0, 11)
    _mads  = []
    for _lo2, _hi2 in zip(_be[:-1], _be[1:]):
        _m = (_preds >= _lo2) & (_preds < _hi2)
        if _m.sum() > 0:
            _mads.append(abs(float(_preds[_m].mean()) - float(_trues[_m].mean())))
    _cal_err = float(_np2.mean(_mads)) if _mads else 1.0

    _sweep_results.append({
        "arch":      _dims,
        "vlw":       _vlw,
        "val_loss":  _best_val,
        "cal_err":   _cal_err,
        "net":       _net,
    })
    print(f"  val_combined={_best_val:.4f}  cal_err={_cal_err:.4f}")

# Sort by val combined loss
_sweep_results.sort(key=lambda r: r["val_loss"])

# Print results table
_w = 64
print("\n" + "─" * _w)
print(f"{'arch':>14}  {'vlw':>5}  {'val_loss':>9}  {'cal_err':>8}  ")
print("─" * _w)
for _r in _sweep_results:
    _marker = "  ← best" if _r is _sweep_results[0] else ""
    print(f"  {str(_r['arch']):>12}  {_r['vlw']:>5}  {_r['val_loss']:>9.4f}  {_r['cal_err']:>8.4f}{_marker}")
print("─" * _w)

# Promote the best config to _new_joint (used by cell 4.5)
_best_cfg = _sweep_results[0]
_new_joint = _best_cfg["net"]

# Save versioned checkpoint
import json as _json2
_manifest_path_sw = DATA_DIR / "self_play" / "manifest.json"
_iter_sw = 0
if _manifest_path_sw.exists():
    with open(_manifest_path_sw) as _fh:
        _iter_sw = len(_json2.load(_fh).get("batches", []))

_sweep_path = DATA_DIR / f"joint_net_iter_{_iter_sw}_sweep_best.pkl"
_new_joint.save(_sweep_path)

print(f"\nBest config: arch={_best_cfg['arch']}  vlw={_best_cfg['vlw']}")
print(f"Saved as {_sweep_path.name}")
print(f"_new_joint updated — proceed to cell 4.5 for promotion evaluation.")

---
## Section 5 — Full Training Loop

**Use this section instead of cycling through Sections 3–4 manually.**

Cell 5.1 wraps the generate → train → validate → evaluate → promote cycle
into a single automated loop. It is designed to be re-run safely: it reads
the current replay buffer manifest to determine which iteration to start from,
so interrupting and resuming always continues from where you left off.

**When to use this vs. Sections 3–4 individually:**
- **Section 5** — standard training; set your parameters and let it run.
- **Section 3–4** — one-off data generation, architecture sweeps (cell 4.6),
  or when you want fine-grained control over a single iteration.


In [ ]:
# 5.1 — Run Complete Training Loop
#
# Automated SP_N_ITERS iterations of: generate games → train joint net →
# validate → evaluate → promote.
#
# Reads the replay buffer manifest to determine the current iteration, so
# re-running this cell safely resumes where you left off.
#
# All parameters below mirror Sections 3.1 and 4.1. Edit them here; changes
# in the other sections have no effect on this cell (section-independent).
#
# Prerequisites: Section 0 must have been run to set DATA_DIR, DB_PATH,
# and DEVICE. fm_model.pkl and matchup_db.pkl must exist in DATA_DIR.

import json   as _json
import time   as _time
import numpy  as _np

# ╔══════════════════════════════════════════════════════════════╗
# ║  5.1 — FULL TRAINING LOOP CONFIGURATION                     ║
# ╠══════════════════════════════════════════════════════════════╣
# ║  Self-Play (SP_) Parameters                                  ║
# ╚══════════════════════════════════════════════════════════════╝

SP_N_GAMES_PER_ITER  = 1000    # Games to generate per iteration.
SP_N_WORKERS         = 4      # Parallel workers (set to CPU core count - 1).
SP_BAN_PROBABILITY   = 0.35   # Probability that each top brawler gets banned.
SP_MAX_BANS          = 6      # Max total bans per game.
SP_N_SIMS_ITER0      = 2000   # Sims per pick in iteration 0 (no prior yet).
SP_N_SIMS_ITER_N     = 5000   # Sims per pick in iteration 1+ (rollout-free mode).
SP_UCB1_C            = 0.5    # MCTS exploration constant. Range: 0.3–1.5.
SP_PUCT_ALPHA        = 0.7    # PUCT prior weight. Range: 0.3–0.9.
SP_MIN_PICK_RATE     = 0.005  # Brawlers below this pick rate are excluded.
SP_N_ITERS           = 5      # Number of iterations to run in this cell.

# ╔══════════════════════════════════════════════════════════════╗
# ║  Joint Network (JN_) Parameters                             ║
# ╚══════════════════════════════════════════════════════════════╝

JN_HIDDEN_DIMS       = [256, 128]  # Shared trunk hidden layer sizes.
JN_LR                = 1e-3        # Learning rate.
JN_WEIGHT_DECAY      = 1e-4        # L2 regularization.
JN_BATCH_SIZE        = 512         # Mini-batch size.
JN_MAX_EPOCHS        = 50          # Hard epoch cap.
JN_PATIENCE          = 7           # Early stopping patience.
JN_POLICY_WEIGHT     = 1.0         # Policy cross-entropy loss weight.
JN_VALUE_WEIGHT      = 1.0         # Value BCE loss weight.
JN_VAL_FRAC          = 0.1         # Fraction of games held out for validation.

# ╔══════════════════════════════════════════════════════════════╗
# ║  Promotion / Evaluation Parameters                          ║
# ╚══════════════════════════════════════════════════════════════╝

LOOP_PROMOTION_THR   = 0.51  # New net must beat this win rate to be promoted.
LOOP_N_EVAL_GAMES    = 200    # Head-to-head evaluation games per iteration.
LOOP_N_EVAL_SIMS     = 500    # Sims per pick during evaluation.
LOOP_EVAL_MIN_RATE   = 0.005  # Min pick rate filter during evaluation.
LOOP_EVAL_PUCT_ALPHA = 0.7    # PUCT alpha during evaluation.

# ── Section-independent imports (reload everything from disk) ──────────────────
from workflow_utils  import run_eval_games, validate_joint_net
from self_play       import run_self_play_batch, load_map_mode_pairs
from joint_net       import train_joint, JointNetInference
from feature_engineering import FeatureSchema

_manifest_path  = DATA_DIR / "self_play" / "manifest.json"
_map_mode_pairs = load_map_mode_pairs(
    DB_PATH,
    known_maps=set(FeatureSchema.load(DATA_DIR / "feature_schema.pkl").maps),
)

_summary = []

print(f"Starting training loop: {SP_N_ITERS} iterations")
print(f"  Data dir       : {DATA_DIR}")
print(f"  Architecture   : {JN_HIDDEN_DIMS}")
print(f"  Promotion thr  : {LOOP_PROMOTION_THR}")
print("─" * 70)

for _step in range(SP_N_ITERS):

    # ── Determine current iteration from manifest ─────────────────────────────
    _iter_num       = 0
    _start_game_idx = 0
    if _manifest_path.exists():
        with open(_manifest_path) as _fh:
            _cur_manifest = _json.load(_fh)
        _batches        = _cur_manifest.get("batches", [])
        _iter_num       = len(_batches)
        _start_game_idx = sum(b.get("n_records", 0) for b in _batches) // 6

    _joint_path = DATA_DIR / "joint_net.pkl"
    _n_sims     = SP_N_SIMS_ITER_N if _joint_path.exists() else SP_N_SIMS_ITER0

    print(f"\n[Iter {_iter_num}]  Step {_step + 1}/{SP_N_ITERS}")
    print(f"  Self-play  : {SP_N_GAMES_PER_ITER} games  |  {_n_sims} sims/pick  |  {SP_N_WORKERS} workers")

    # ── 1. Generate self-play games ───────────────────────────────────────────
    _buf          = ReplayBuffer.load(DATA_DIR / "self_play")
    _games_before = _buf.n_games

    _sp_t0 = _time.perf_counter()
    run_self_play_batch(
        n_games         = SP_N_GAMES_PER_ITER,
        data_dir        = DATA_DIR,
        map_mode_pairs  = _map_mode_pairs,
        replay_buffer   = _buf,
        n_workers       = SP_N_WORKERS,
        start_game_idx  = _start_game_idx,
        seed            = _iter_num,
        n_sims_per_pick = _n_sims,
        min_pick_rate   = SP_MIN_PICK_RATE,
        ban_p           = SP_BAN_PROBABILITY,
        max_bans        = SP_MAX_BANS,
    )
    _sp_elapsed  = _time.perf_counter() - _sp_t0
    _games_added = _buf.n_games - _games_before
    _total_games = _buf.n_games

    _secs_per_game = _sp_elapsed / max(_games_added, 1)
    print(f"  Generated  : {_games_added:,} games in {_sp_elapsed:.1f}s  "
          f"({60.0 / _secs_per_game:.1f} games/min)  —  buffer: {_total_games:,} total")

    # ── 2. Train joint network ────────────────────────────────────────────────
    _records = _buf.all_records()
    _schema  = FeatureSchema.load(DATA_DIR / "feature_schema.pkl")

    _loss_history = []
    _jn_t0        = _time.perf_counter()
    _new_joint    = train_joint(
        records           = _records,
        schema            = _schema,
        lambda_v          = JN_VALUE_WEIGHT,
        lr                = JN_LR,
        weight_decay      = JN_WEIGHT_DECAY,
        batch_size        = JN_BATCH_SIZE,
        max_epochs        = JN_MAX_EPOCHS,
        patience          = JN_PATIENCE,
        val_game_fraction = JN_VAL_FRAC,
        trunk_hidden      = tuple(JN_HIDDEN_DIMS),
        device            = DEVICE,
        verbose           = False,
        loss_history      = _loss_history,
    )
    _jn_elapsed = _time.perf_counter() - _jn_t0

    # Save versioned checkpoint
    _new_joint.save(DATA_DIR / f"joint_net_iter_{_iter_num}.pkl")

    # Best val epoch metrics for the summary table
    _best_ep  = min(_loss_history, key=lambda t: t[3]) if _loss_history else (0, 0.0, 0.0, 0.0)
    _best_pol = _best_ep[1]   # policy component at best val epoch
    _best_val = _best_ep[2]   # value component at best val epoch

    print(f"  Trained    : {len(_loss_history)} epochs in {_jn_elapsed:.1f}s  "
          f"(best-val: pol={_best_pol:.4f}  val={_best_val:.4f})")

    # ── 3. Validate ───────────────────────────────────────────────────────────
    _vr       = validate_joint_net(
        joint_net         = _new_joint,
        records           = _records,
        val_game_fraction = JN_VAL_FRAC,
    )
    _cal_err  = _vr["cal_err"]
    _cal_pass = _vr["cal_pass"]
    _pol_top1 = _vr["pol_top1"]
    _pol_pass = _vr["pol_pass"]
    print(f"  Validation : cal_err={_cal_err:.4f} ({'✓' if _cal_pass else '✗'})  "
          f"pol_top1={_pol_top1}/5 ({'✓' if _pol_pass else '✗'})")

    # ── 4. Promotion evaluation ───────────────────────────────────────────────
    _baseline_joint = None
    if _joint_path.exists():
        _baseline_joint = JointNetInference.load(_joint_path)

    _eval_result = run_eval_games(
        new_joint      = _new_joint,
        baseline_joint = _baseline_joint,
        data_dir       = DATA_DIR,
        db_path        = DB_PATH,
        n_games        = LOOP_N_EVAL_GAMES,
        n_sims         = LOOP_N_EVAL_SIMS,
        min_pick_rate  = LOOP_EVAL_MIN_RATE,
        puct_alpha     = LOOP_EVAL_PUCT_ALPHA,
        seed           = _iter_num,
    )
    _win_rate = _eval_result["new_net_win_rate"]
    _promoted = _win_rate >= LOOP_PROMOTION_THR

    if _promoted:
        _new_joint.save(DATA_DIR / "joint_net.pkl")
        print(f"  Eval       : win_rate={_win_rate:.4f}  → PROMOTED ✓")
    else:
        print(f"  Eval       : win_rate={_win_rate:.4f}  → not promoted  "
              f"(thr={LOOP_PROMOTION_THR})")

    _summary.append({
        "iter":     _iter_num,
        "games":    _total_games,
        "pol_loss": _best_pol,
        "val_loss": _best_val,
        "cal_err":  _cal_err,
        "win_rate": _win_rate,
        "promoted": _promoted,
    })

# ── Summary table ─────────────────────────────────────────────────────────────
print()
print("─" * 70)
print("Training Loop Summary")
print("─" * 70)
print(f"  {'Iter':>4}  {'Games':>6}  {'Train Pol':>9}  {'Train Val':>9}  "
      f"{'Cal Err':>8}  {'Eval Win':>9}  {'Promoted':>8}")
print("  " + "─" * 62)
for _row in _summary:
    _promo_mark = "✓" if _row["promoted"] else "✗"
    print(
        f"  {_row['iter']:>4}  {_row['games']:>6,}  "
        f"{_row['pol_loss']:>9.4f}  {_row['val_loss']:>9.4f}  "
        f"{_row['cal_err']:>8.4f}  {_row['win_rate']:>9.4f}  {_promo_mark:>8}"
    )
print("─" * 70)
_n_promoted = sum(1 for _r in _summary if _r["promoted"])
print(f"  {_n_promoted}/{len(_summary)} iterations promoted the joint network.")


---
## Section 6 — Final Benchmarks

Run these two cells after the full training loop is complete.

- **Cell 6.1** measures the per-pick speed gain from rollout-free MCTS so you
  can choose a production simulation budget that fits your hardware.
- **Cell 6.2** prints a concise reference card summarising all trained
  artifacts and the recommended live configuration.


In [ ]:
# 6.1 — Rollout-Free Speed Gain
#
# Compares per-pick MCTS wall-clock time in two modes:
#   1. FM-only baseline  — every leaf node runs a full FM rollout to estimate value.
#   2. Joint-net mode    — the value head replaces rollouts entirely; each
#                          simulation is a forward pass through the network trunk.
#
# Tests at sim counts [500, 1000, 2000, 5000, 10000]. Prints time per pick and
# speedup factor. Use the output to pick a production n_simulations that gives
# you the quality you want within the latency budget of a live draft.
#
# Prerequisites: fm_model.pkl, matchup_db.pkl must exist in DATA_DIR.
#   joint_net.pkl is optional — if absent, only the FM baseline column is shown.
#
# This cell reloads all dependencies from disk and is section-independent.

from workflow_utils import benchmark_rollout_free_speed

_speed_results = benchmark_rollout_free_speed(
    data_dir      = DATA_DIR,
    db_path       = DB_PATH,
    sim_counts    = [500, 1000, 2000, 5000, 10000],
    n_warmup      = 1,
    n_timed       = 3,
    ucb1_c        = 0.5,
    puct_alpha    = 0.7,
    min_pick_rate = 0.005,
)


In [ ]:
# 6.2 — Final Configuration Summary
#
# Reads all trained artifacts from DATA_DIR and prints a reference card for
# deploying the agent. Paste the "Recommended live config" block into
# draft_playground.ipynb when you load the agent.
#
# Metrics shown here are derived from artifact files on disk (no recomputation).
# For FM and value-head quality metrics, refer to cells 1.5 and 4.4 / the
# Section 5 summary table.
#
# This cell is section-independent.

import json as _json

_w = 54

# ── Artifact inventory ────────────────────────────────────────────────────────
_fm_ok     = (DATA_DIR / "fm_model.pkl").exists()
_schema_ok = (DATA_DIR / "feature_schema.pkl").exists()
_db_ok     = (DATA_DIR / "matchup_db.pkl").exists()
_jn_ok     = (DATA_DIR / "joint_net.pkl").exists()

# ── Self-play statistics from manifest ───────────────────────────────────────
_manifest_path = DATA_DIR / "self_play" / "manifest.json"
_n_iters       = 0
_total_games   = 0
_total_records = 0

if _manifest_path.exists():
    with open(_manifest_path) as _fh:
        _manifest = _json.load(_fh)
    _batches       = _manifest.get("batches", [])
    _n_iters       = len(_batches)
    _total_records = sum(b.get("n_records", 0) for b in _batches)
    _total_games   = _total_records // 6

# ── Recommended production budget from 6.1 (if available) ────────────────────
try:
    _rec_sims = next(
        r["n_sims"] for r in _speed_results
        if r.get("joint_sec_per_pick") is not None and r["joint_sec_per_pick"] <= 5.0
    )
    _rec_note = f"{_rec_sims:,}  (rollout-free, ≤5s per pick on this hardware)"
except (NameError, StopIteration):
    _rec_sims = 2000
    _rec_note = f"{_rec_sims:,}  (default — run cell 6.1 for hardware-tuned advice)"

# ── Print reference card ──────────────────────────────────────────────────────
print("=" * _w)
print(f"  Trained Agent Summary — {SEASON}")
print("=" * _w)

print(f"\nArtifacts in {DATA_DIR}")
_mark = lambda ok: "✓" if ok else "✗"
print(f"  [{_mark(_fm_ok)}]  fm_model.pkl")
print(f"  [{_mark(_schema_ok)}]  feature_schema.pkl")
print(f"  [{_mark(_db_ok)}]  matchup_db.pkl")
print(f"  [{_mark(_jn_ok)}]  joint_net.pkl")

print(f"\nSelf-play statistics")
print(f"  Iterations completed  : {_n_iters}")
print(f"  Total games generated : {_total_games:,}")
print(f"  Total records         : {_total_records:,}")

print(f"\nQuality metrics")
print(f"  FM val log-loss       : see cell 1.5")
print(f"  FM calibration error  : see cell 1.5")
print(f"  Value cal. error      : see cell 4.4 / Section 5 summary table")
print(f"  Policy top-1 agree.   : see cell 4.4 / Section 5 summary table")

print(f"\nRecommended live configuration")
print(f"  n_simulations = {_rec_note}")
print(f"  ucb1_c        = 0.5")
print(f"  puct_alpha    = 0.7")
print(f"  min_pick_rate = 0.005")

print(f"\nTo load the agent:")
print(f"  from joint_net import JointNetInference")
print(f"  joint_net = JointNetInference.load(DATA_DIR / \"joint_net.pkl\")")

print()
print("=" * _w)

if not _jn_ok:
    print("\n⚠  joint_net.pkl is missing. Run Sections 4–5 to train and promote")
    print("   a joint network before deploying the agent.")


---
## Section 7 — Next Steps

### Using the trained agent

Open `draft_playground.ipynb` to use the agent interactively. Load the
artifacts with:

```python
from joint_net import JointNetInference
joint_net = JointNetInference.load(DATA_DIR / "joint_net.pkl")
```

Use the recommended `n_simulations` from cell 6.1 as your starting point.

### Adding more self-play iterations

The replay buffer is cumulative — each new iteration appends to what is
already there. To continue training:

- **Section 5, cell 5.1** — the easiest path: re-run cell 5.1. It reads the
  manifest and resumes from the current iteration automatically.
- **Sections 3–4 manually** — run cell 3.4 to generate games, then
  cells 4.2 → 4.3 → 4.4 → 4.5 to train and evaluate. Use this when you
  want fine-grained control (e.g., architecture sweeps via cell 4.6).

### New season data

A new season requires retraining from scratch — brawler balance shifts
change the FM labels, so the old network is no longer valid. Re-run
starting from Section 1 with the new `DB_PATH` and a fresh `DATA_DIR`.

### Improving a stalled agent

If the agent stops being promoted after several iterations, consider:

- **Lower `LOOP_PROMOTION_THR`** (e.g., 0.525 → 0.51) if win rates are
  consistently just below the threshold — this may be measurement noise.
- **Run the architecture sweep** (cell 4.6) after accumulating ≥ 3k games
  to find a better model capacity for your data size.
- **Increase `SP_N_GAMES_PER_ITER`** — larger batches give the network
  more signal per training round and typically reduce noise in promotion eval.
- **Increase `SP_N_SIMS_ITER_N`** — higher simulation budgets produce
  better MCTS visit distributions as policy targets.
